# DynaGuard Inference Pipeline
### 100-Turn Compliance Evaluation via `dynaguard_chat_messages`
**Kaggle & Google Colab Compatible**

A lightweight, standalone, and transparent inference notebook for evaluating **DynaGuard** (8B / 4B / 1.7B) guardian models on structured chat message test cases in `dynaguard_chat_messages/`.

**Key Pipeline Principles:**
- **Exact Official Chat Template**: Passes the multi-turn dialogue message array directly to `tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, ...)` as demonstrated in the DynaGuard documentation.
- **Sequential Processing**: Evaluates test cases one by one directly from `dynaguard_chat_messages/` without accumulating activations in GPU memory.
- **Critical Transparent Decoding**: Slices only newly generated tokens (`outputs[0][inputs['input_ids'].shape[-1]:]`) and preserves all raw model tokens verbatim without parsing, stripping XML, or modifying text.
- **Resume & Reproducibility**: Skips existing outputs on re-run, saves sent prompts to `sent_prompts/`, and logs execution metadata to `logs/run_log.jsonl`.

## 1. Environment Detection & Dependencies

In [ ]:
import os, sys, json, re, time, datetime

# Detect execution environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

platform_name = 'Google Colab' if IS_COLAB else ('Kaggle' if IS_KAGGLE else 'Local / Standalone')
print(f"Detected Platform: {platform_name}")

# If running in Colab and repository is not cloned, clone it
if IS_COLAB and not os.path.exists('dynaguard_chat_messages') and not os.path.exists('DynaGuard'):
    !git clone https://github.com/AkarshiAaryan/DynaGuard.git
    %cd DynaGuard

# Install required dependencies
!pip install -q -U transformers accelerate

## 2. Hardware Detection & Device Setup

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"bfloat16 supported: {bf16_supported}")
    # Use bfloat16 on Ampere/Hopper (A100, L4) or float16 on Turing (T4)
    TORCH_DTYPE = torch.bfloat16 if bf16_supported else torch.float16
    DEVICE_MAP = 'auto'
else:
    print("No CUDA GPU detected. Falling back to CPU.")
    TORCH_DTYPE = torch.float32
    DEVICE_MAP = None

print(f"Selected Precision: {TORCH_DTYPE}, Device Map: {DEVICE_MAP}")

## 3. Directory & Input Configuration
Uses `dynaguard_chat_messages/` containing the 100 benchmark test cases structured as JSON chat message arrays.

In [ ]:
# Determine working base directory
if IS_KAGGLE:
    BASE_DIR = '/kaggle/working'
elif IS_COLAB:
    BASE_DIR = os.getcwd()
else:
    BASE_DIR = '.'

# Locate dynaguard_chat_messages directory
candidates = [
    'dynaguard_chat_messages',
    os.path.join(BASE_DIR, 'dynaguard_chat_messages'),
    os.path.join(BASE_DIR, 'DynaGuard', 'dynaguard_chat_messages'),
    'dynaguard_inputs',
    os.path.join(BASE_DIR, 'dynaguard_inputs'),
]
INPUT_DIR = next((d for d in candidates if os.path.isdir(d)), 'dynaguard_chat_messages')

OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
SENT_PROMPTS_DIR = os.path.join(BASE_DIR, 'sent_prompts')
LOGS_DIR = os.path.join(BASE_DIR, 'logs')
RUN_LOG_PATH = os.path.join(LOGS_DIR, 'run_log.jsonl')

for d in [OUTPUT_DIR, SENT_PROMPTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Model selection:
# - 'tomg-group-umd/DynaGuard-8B' (Flagship 8B model)
# - 'tomg-group-umd/DynaGuard-4B' (Fast & low-memory, ideal for 15GB T4 GPUs)
# - 'tomg-group-umd/DynaGuard-1.7B' (Ultra-lightweight)
MODEL_ID = os.environ.get('DYNAGUARD_MODEL_ID', 'tomg-group-umd/DynaGuard-8B')

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0  # Greedy decoding for deterministic classification
TOP_P = 0.95
DO_SAMPLE = False

print(f"Active Model ID  : {MODEL_ID}")
print(f"Input Directory  : {INPUT_DIR} (exists: {os.path.isdir(INPUT_DIR)})")
print(f"Output Directory : {OUTPUT_DIR}")

## 4. Prompt Analysis: Inspecting the DynaGuard Chat Message Format
Examines the structure of `TC01_student_own_grade.json` and previews the raw message array.

In [ ]:
sample_case_path = os.path.join(INPUT_DIR, 'TC01_student_own_grade.json')
if os.path.exists(sample_case_path):
    with open(sample_case_path, 'r', encoding='utf-8') as f:
        sample_messages = json.load(f)
    
    print("=" * 75)
    print(f"SAMPLE TEST CASE: TC01_student_own_grade.json ({len(sample_messages)} turns)")
    print("=" * 75)
    for idx, turn in enumerate(sample_messages, start=1):
        role = turn['role'].upper()
        content = turn['content']
        print(f"Turn {idx} [{role}]:\n{content}\n")
else:
    print(f"Sample file not found at {sample_case_path}")

## 5. Load DynaGuard Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading model and tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model_kwargs = {'torch_dtype': TORCH_DTYPE, 'trust_remote_code': True}
if DEVICE_MAP is not None:
    model_kwargs['device_map'] = DEVICE_MAP

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
print("Model loaded successfully!")

# Print formatted string produced by tokenizer.apply_chat_template
if os.path.exists(sample_case_path):
    formatted_prompt = tokenizer.apply_chat_template(sample_messages, add_generation_prompt=True, tokenize=False)
    print("\n" + "=" * 75)
    print("EXACT FORMATTED PROMPT STREAM APPLIED TO MODEL (WITH GENERATION PROMPT):")
    print("=" * 75)
    print(formatted_prompt)
    print("=" * 75)

## 6. Single-Case Verification Test
Runs `TC01_student_own_grade.json` through the model to observe the raw verdict.

In [ ]:
if os.path.exists(sample_case_path):
    inputs = tokenizer.apply_chat_template(
        sample_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    
    start_t = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.perf_counter() - start_t
    
    prompt_length = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=False)
    print(f"Inference Time: {elapsed:.2f}s")
    print("Raw Model Generated Response:")
    print("-" * 50)
    print(response)
    print("-" * 50)

## 7. Sequential Batch Evaluation Over All 100 Test Cases
Processes each test case in `dynaguard_chat_messages/` one-by-one in natural numerical order (`TC01` to `TC100`), writes raw outputs verbatim, and logs metrics.

In [ ]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

# Collect all test cases (.json files representing message arrays)
test_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.json') and f != 'dynaguard_messages.json']
if not test_files:
    test_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]

test_files.sort(key=natural_sort_key)
total_cases = len(test_files)
print(f"Found {total_cases} test case(s) in {INPUT_DIR}.")
if total_cases > 0:
    print(f"Range: {test_files[0]}  -->  {test_files[-1]}")

RERUN_EXISTING = False  # Set to True to re-evaluate already completed cases

successful_count = 0
failed_count = 0
skipped_count = 0

for idx, filename in enumerate(test_files, start=1):
    stem = os.path.splitext(filename)[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_output.txt")
    prompt_path = os.path.join(SENT_PROMPTS_DIR, f"{stem}_prompt.txt")
    in_path = os.path.join(INPUT_DIR, filename)

    # Resume logic: skip existing non-empty outputs
    if not RERUN_EXISTING and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"[{idx:03d}/{total_cases:03d}] {filename} -> SKIPPED (already evaluated)")
        skipped_count += 1
        continue

    print(f"[{idx:03d}/{total_cases:03d}] Processing {filename} ... ", end='', flush=True)

    try:
        # Load test case content
        if filename.endswith('.json'):
            with open(in_path, 'r', encoding='utf-8') as f:
                messages = json.load(f)
        else:
            with open(in_path, 'r', encoding='utf-8') as f:
                txt_content = f.read()
            messages = [{"role": "user", "content": txt_content}]

        # Apply official chat template with generation prompt
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        # Archive verbatim prompt for reproducibility
        sent_prompt_str = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)
        with open(prompt_path, 'w', encoding='utf-8') as pf:
            pf.write(sent_prompt_str)

        # Execute generation
        prompt_length = inputs["input_ids"].shape[-1]
        start_t = time.perf_counter()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                pad_token_id=tokenizer.pad_token_id,
            )
        elapsed_t = time.perf_counter() - start_t

        # CRITICAL: Decode only newly generated tokens verbatim
        raw_generated = tokenizer.decode(
            outputs[0][prompt_length:],
            skip_special_tokens=False,
        )

        # Save raw model response without any parsing
        with open(out_path, 'w', encoding='utf-8') as of:
            of.write(raw_generated)

        # Log record
        log_entry = {
            'input_filename': filename,
            'output_filename': f"{stem}_output.txt",
            'status': 'success',
            'inference_time': round(elapsed_t, 4),
            'error_message': None,
            'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        }
        with open(RUN_LOG_PATH, 'a', encoding='utf-8') as lf:
            lf.write(json.dumps(log_entry) + '\n')

        print(f"DONE ({elapsed_t:.2f}s)")
        successful_count += 1

    except Exception as exc:
        print(f"FAILED: {exc}")
        failed_count += 1
        log_entry = {
            'input_filename': filename,
            'output_filename': f"{stem}_output.txt",
            'status': 'failed',
            'inference_time': 0.0,
            'error_message': str(exc),
            'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        }
        with open(RUN_LOG_PATH, 'a', encoding='utf-8') as lf:
            lf.write(json.dumps(log_entry) + '\n')

print("\n" + "=" * 70)
print(f"Inference Run Complete: {successful_count} succeeded, {failed_count} failed, {skipped_count} skipped.")
print(f"Outputs saved to: {OUTPUT_DIR}")
print(f"Sent prompts to : {SENT_PROMPTS_DIR}")
print(f"Log saved to    : {RUN_LOG_PATH}")
print("=" * 70)

## 8. Summary Inspection

In [ ]:
import pandas as pd

# Load and display run log records
if os.path.exists(RUN_LOG_PATH):
    records = []
    with open(RUN_LOG_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    
    df = pd.DataFrame(records)
    print(f"Total logged inferences: {len(df)}")
    display(df.tail(20))
else:
    print("No log file found yet.")

# Preview first 3 raw outputs
out_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('_output.txt')], key=natural_sort_key)
print(f"\nTotal output files saved: {len(out_files)}")
for f in out_files[:3]:
    print(f"\n=== {f} ===")
    with open(os.path.join(OUTPUT_DIR, f), 'r', encoding='utf-8') as fh:
        print(fh.read())
    print("=" * 40)